# Model 2: SGDClassifier (Perceptron / Soft Threshold)

**Project:** DataCo Smart Supply Chain — Late Delivery Risk Classification  
**Course:** CS280/CS485 Introduction to Artificial Intelligence — Spring 2026  
**Notebook:** `03_model_sgd.ipynb` — implements Section 5.2 of `docs.md`  

This notebook trains and evaluates a `SGDClassifier` with `loss='modified_huber'` — the scikit-learn
implementation of the soft-threshold Perceptron from Lecture 4. It loads the prepared data produced
by Phase 0, tunes hyperparameters via `GridSearchCV`, evaluates on the held-out test set, and saves
all results to `results/sgd_results.pkl` for use in Phase 6 (merge & compare).

## 1. Imports

All dependencies are standard sklearn / scientific Python — no installation required beyond `requirements.txt`.

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Imports OK')

## 2. Load Prepared Data

We load exclusively from `artifacts/prepared_data.pkl` — the single source of truth produced by
Phase 0. **No raw CSV is accessed here**, and no preprocessing objects (scaler, encoder) are refit.
This guarantees identical train/test splits across all model phases.

In [ ]:
ARTIFACT_PATH = '../artifacts/prepared_data.pkl'
RESULTS_PATH  = '../results/sgd_results.pkl'

with open(ARTIFACT_PATH, 'rb') as f:
    data = pickle.load(f)

X_train        = data['X_train']
X_test         = data['X_test']
y_train        = data['y_train']
y_test         = data['y_test']
feature_names  = data['feature_names']

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_train class balance : {np.bincount(y_train)}')
print(f'y_test  class balance : {np.bincount(y_test)}')
print(f'Number of features    : {len(feature_names)}')

**Interpretation:** The shapes confirm we are working with the preprocessed, scaled training and test
sets from Phase 0. The class balance is checked to confirm stratified splitting preserved the
original distribution. The feature matrix is already scaled by `RobustScaler`, which is essential
for the SGDClassifier: weight updates are proportional to feature magnitude, so unscaled features
would bias learning toward high-magnitude columns.

## 3. Lecture 4 Connection — Perceptron, Soft Threshold & Regularization

### 3.1 Hard Threshold (Standard Perceptron — Lecture 4)

The standard Perceptron from Lecture 4 applies a **hard threshold** (step function) to the dot
product of weights and inputs:

$$h_{\mathbf{w}}(\mathbf{x}) = \begin{cases} 1 & \text{if } \mathbf{w} \cdot \mathbf{x} \geq 0 \\ 0 & \text{otherwise} \end{cases}$$

This function is **non-differentiable** at zero, so gradient-based optimisation cannot be applied
directly, and it cannot output calibrated probabilities — only hard class labels.

### 3.2 Soft Threshold Motivation (Lecture 4)

Lecture 4 motivates replacing the step function with a **smooth, sigmoid-like approximation** that:
- Is differentiable everywhere → gradient descent can minimise the loss
- Outputs values in (0, 1) → interpretable as class probabilities
- Reduces to near-0/1 predictions far from the decision boundary, matching the hard threshold in the
  limit

`SGDClassifier(loss='modified_huber')` is the scikit-learn implementation of this concept:
the **modified Huber loss** combines the robustness of the hinge loss with the probability output of
logistic regression, making it the closest sklearn equivalent to the soft-threshold Perceptron.

### 3.3 Perceptron Learning Rule (Lecture 4)

The weight update rule from Lecture 4:

$$w_i \leftarrow w_i + \alpha\,(y - h_{\mathbf{w}}(\mathbf{x})) \cdot x_i$$

where:
- $\alpha$ (the `eta0` parameter in sklearn) is the **step size** — how aggressively weights are
  updated at each iteration
- $(y - h_{\mathbf{w}}(\mathbf{x}))$ is the prediction error — if the prediction is correct, the
  update is zero
- $x_i$ scales the update by the feature value — larger features cause larger weight shifts
  (another reason scaling is critical)

### 3.4 Regularization — Mapping `alpha` to λ

Lecture 4's regularization framework:

$$\text{cost}(h) = \text{loss}(h) + \lambda \cdot \text{complexity}(h)$$

In `SGDClassifier`, the `alpha` hyperparameter **is** λ: it is the coefficient of the L2 penalty
added to the per-sample loss:

$$\text{cost} = \frac{1}{n}\sum_{i=1}^{n} \ell(y_i, h_{\mathbf{w}}(\mathbf{x}_i)) + \alpha \cdot \|\mathbf{w}\|^2$$

- **Large `alpha` (large λ):** strong regularization → smaller weights → simpler model → less
  overfitting, but may underfit
- **Small `alpha` (small λ):** weak regularization → weights can grow freely → more complex
  decision boundary → may overfit

The `eta0` parameter corresponds to the step-size α in the perceptron learning rule above.
Note the naming collision: sklearn's `alpha` = lecture's λ (regularization), while sklearn's
`eta0` = lecture's α (learning rate).

## 4. Hyperparameter Grid

The grid follows Section 5.2 of `docs.md` exactly:

| Parameter | Values | Meaning |
|:---|:---|:---|
| `alpha` | {0.0001, 0.001, 0.01} | L2 regularization strength (= λ in lecture formula) |
| `eta0` | {0.001, 0.01, 0.1} | Learning rate step size (= α in perceptron rule) |
| `max_iter` | {500, 1000} | Maximum training epochs |

`learning_rate='constant'` is **required** so that `eta0` takes effect. The default
`learning_rate='optimal'` ignores `eta0` and computes its own schedule, which would make `eta0`
a dead hyperparameter in the grid.

In [ ]:
param_grid = {
    'alpha'    : [0.0001, 0.001, 0.01],
    'eta0'     : [0.001, 0.01, 0.1],
    'max_iter' : [500, 1000],
}

total_fits = (len(param_grid['alpha']) *
              len(param_grid['eta0'])  *
              len(param_grid['max_iter']) * 5)  # 5-fold CV

print(f'Grid size          : {len(param_grid["alpha"]) * len(param_grid["eta0"]) * len(param_grid["max_iter"])} combinations')
print(f'Total model fits   : {total_fits}  (combinations × 5 CV folds)')

## 5. GridSearchCV — Hyperparameter Tuning

We use `GridSearchCV` with 5-fold cross-validation and `scoring='f1'` (per docs.md and PHASES.md).
F1 is chosen over accuracy because the dataset has class imbalance: accuracy can be high even when
the model misses most late deliveries (false negatives), which is the costly error in this domain.

The base estimator is fixed as:
- `loss='modified_huber'` — enables `predict_proba()` and implements the soft threshold
- `learning_rate='constant'` — ensures `eta0` is the actual step size used
- `random_state=42` — reproducibility

In [ ]:
base_sgd = SGDClassifier(
    loss='modified_huber',
    learning_rate='constant',
    random_state=42
)

grid_search = GridSearchCV(
    estimator  = base_sgd,
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'f1',
    n_jobs     = -1,
    verbose    = 1,
    refit      = True   # refit best model on full X_train after CV
)

print('Fitting GridSearchCV ...')
grid_search.fit(X_train, y_train)
print('Done.')

## 6. Best Parameters

The grid search selects the hyperparameter combination that maximises mean 5-fold F1 on the
training set. The best estimator has been refit on the full `X_train` by `GridSearchCV`.

In [ ]:
best_params    = grid_search.best_params_
best_cv_score  = grid_search.best_score_
best_estimator = grid_search.best_estimator_

print('Best hyperparameters:')
for k, v in best_params.items():
    print(f'  {k:12s} = {v}')
print(f'\nBest CV F1 (mean over 5 folds) : {best_cv_score:.4f}')

**Interpretation:** The best `alpha` (λ in lecture notation) indicates the regularization strength
that best balances bias and variance on the training folds. A small `alpha` allows the model more
freedom to fit complex patterns; a large `alpha` shrinks weights toward zero. The best `eta0`
reflects the optimal learning rate for the SGD update rule on this feature space.

## 7. Predictions & Test-Set Metrics

The best estimator (refit on full `X_train`) is applied once to the held-out `X_test`. This test
set was never seen during training or CV — it is a genuine measure of generalisation.

In [ ]:
y_pred  = best_estimator.predict(X_test)
y_proba = best_estimator.predict_proba(X_test)[:, 1]

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
roc_auc   = roc_auc_score(y_test, y_proba)

# Train F1 for overfitting chart in Phase 6
y_train_pred = best_estimator.predict(X_train)
train_f1     = f1_score(y_train, y_train_pred, zero_division=0)
test_f1      = f1

print('=== Test-Set Metrics ===')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print()
print(f'  Train F1  : {train_f1:.4f}  (for overfitting chart)')
print(f'  Test  F1  : {test_f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['On-Time (0)', 'Late (1)']))

**Interpretation:** The SGDClassifier is a **linear model** — it learns a single hyperplane to
separate the two classes. Supply chain late-delivery prediction is unlikely to be perfectly linearly
separable, so we expect moderate F1. The gap between Train F1 and Test F1 reveals overfitting:
if Train F1 >> Test F1, the regularization `alpha` should be increased (larger λ in the lecture
formula). Recall on the Late (1) class is the most business-critical metric: missing a late
delivery (false negative) is more costly than a false alarm.

## 8. Confusion Matrix

The confusion matrix breaks down predictions into True Positives (correctly flagged late),
True Negatives (correctly cleared on-time), False Positives (false alarms), and False Negatives
(missed late deliveries — the most costly error).

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['On-Time (0)', 'Late (1)'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — SGDClassifier (Modified Huber)', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('../results/sgd_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nRaw confusion matrix:\n{cm}')
print(f'TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}')

**Interpretation:** A well-performing model for this task should minimise the **False Negatives
(FN)** cell — bottom-left of the matrix. Each FN represents a late delivery that the model failed
to flag, giving logistics teams no opportunity to intervene. False Positives (top-right) generate
unnecessary alerts but have a lower business cost. The confusion matrix quantifies this tradeoff
concretely.

## 9. 5-Fold Stratified Cross-Validation

We run 5-fold stratified CV on the **training set** with the best estimator to assess score
stability. High variance across folds indicates sensitivity to the specific data split; low variance
indicates a robust model. This CV is computed after the grid search (not part of it) to provide an
independent stability estimate.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_f1_scores = cross_val_score(
    best_estimator, X_train, y_train,
    cv=cv_strategy, scoring='f1', n_jobs=-1
)

print('5-Fold Stratified CV F1 scores (training set):')
for i, s in enumerate(cv_f1_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'\n  Mean  : {cv_f1_scores.mean():.4f}')
print(f'  Std   : {cv_f1_scores.std():.4f}')

**Interpretation:** The mean CV F1 on the training set is an in-sample estimate of generalisation.
If this is notably higher than the test F1 reported in Section 7, the model is overfitting — the
regularization penalty (α / `alpha`) should be increased. A low standard deviation across folds
indicates the model performs consistently regardless of the exact data partition.

## 10. CV Score Distribution Plot

Visualising the fold-by-fold scores gives an immediate sense of stability. Wide spread indicates
high variance (sensitivity to data partition); tight spread indicates a stable estimator.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
fold_labels = [f'Fold {i}' for i in range(1, 6)]
bars = ax.bar(fold_labels, cv_f1_scores, color='steelblue', alpha=0.8, edgecolor='white')
ax.axhline(cv_f1_scores.mean(), color='crimson', linestyle='--', linewidth=1.5,
           label=f'Mean F1 = {cv_f1_scores.mean():.4f}')
ax.set_ylim(0, 1)
ax.set_ylabel('F1-Score')
ax.set_title('5-Fold CV F1 Scores — SGDClassifier', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('../results/sgd_cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Each bar is one fold's F1 score on its held-out validation slice. The red
dashed line shows the mean. Consistent bar heights confirm that the model's performance does not
depend strongly on which 20% of training data was held out during validation.

## 11. Summary Table

Condensed view of all key metrics for easy comparison in Phase 6.

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC',
               'Train F1', 'Test F1', 'CV Mean F1', 'CV Std F1'],
    'Value': [
        round(accuracy,  4),
        round(precision, 4),
        round(recall,    4),
        round(f1,        4),
        round(roc_auc,   4),
        round(train_f1,  4),
        round(test_f1,   4),
        round(cv_f1_scores.mean(), 4),
        round(cv_f1_scores.std(),  4),
    ]
})
print(summary.to_string(index=False))

**Interpretation:** This table condenses all evaluation dimensions into one view. The Train/Test F1
pair reveals the bias-variance balance: small gap → well-regularized; large gap → overfit.
ROC-AUC is threshold-independent (measures ranking quality), making it complementary to F1
(which depends on the 0.5 decision threshold). Both metrics appear in the Phase 6 comparison
table alongside the other four models.

## 12. Save Results

All outputs are saved to `results/sgd_results.pkl` using the exact schema defined in PHASES.md §2.3.
Phase 6 loads this file directly — any key mismatch will break the merge notebook. The schema is
verified with an assertion before pickling.

In [ ]:
results = {
    'model_name'  : 'SGDClassifier',
    'model'       : best_estimator,
    'best_params' : best_params,
    'y_pred'      : y_pred,
    'y_proba'     : y_proba,
    'metrics'     : {
        'accuracy'  : accuracy,
        'precision' : precision,
        'recall'    : recall,
        'f1'        : f1,
        'roc_auc'   : roc_auc,
    },
    'cv_f1_scores'      : cv_f1_scores,
    'train_f1'          : train_f1,
    'test_f1'           : test_f1,
    'feature_importance': None,   # not applicable for linear SGDClassifier
}

# Verify schema before pickling
required_keys = {
    'model_name', 'model', 'best_params', 'y_pred', 'y_proba',
    'metrics', 'cv_f1_scores', 'train_f1', 'test_f1', 'feature_importance'
}
assert set(results.keys()) == required_keys, f'Schema mismatch: {set(results.keys()) ^ required_keys}'

required_metric_keys = {'accuracy', 'precision', 'recall', 'f1', 'roc_auc'}
assert set(results['metrics'].keys()) == required_metric_keys, 'Metrics schema mismatch'

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(results, f)

print(f'Results saved to: {RESULTS_PATH}')
print(f'Keys: {list(results.keys())}')
print(f'Metrics: {results["metrics"]}')

**Verification:** The assertions above confirm that the saved dict has exactly the keys required by
PHASES.md §2.3. If either assertion fails, the notebook raises an error before writing the file,
preventing a silently malformed artifact from reaching Phase 6.

`feature_importance` is `None` for the SGDClassifier: while the model does learn a weight vector
(`best_estimator.coef_`), weight magnitude in a linear model is a proxy for importance only when
features are on the same scale (which they are, post-RobustScaler). This can be explored as an
extension but is not required for Phase 6's comparison logic.

## 13. Conclusion

This notebook implemented Section 5.2 of `docs.md`:

- **Model:** `SGDClassifier(loss='modified_huber', learning_rate='constant', random_state=42)` — the
  scikit-learn soft-threshold Perceptron, directly derived from Lecture 4's perceptron concept
- **Tuning:** `GridSearchCV(cv=5, scoring='f1', n_jobs=-1)` over the full `alpha × eta0 × max_iter`
  grid from docs.md
- **Evaluation:** confusion matrix, accuracy / precision / recall / F1 / ROC-AUC on the held-out
  test set, plus train F1 for the Phase 6 overfitting chart, plus 5-fold stratified CV F1
- **Lecture 4 connection:** hard-threshold formula, soft-threshold motivation, perceptron learning
  rule, and mapping of `alpha` → λ in `cost(h) = loss(h) + λ · complexity(h)`
- **Output:** `results/sgd_results.pkl` with the exact schema required by PHASES.md §2.3

**Expected scientific role:** As a linear classifier, the SGDClassifier establishes the linear
baseline. Its performance relative to Random Forest and XGBoost quantifies how much of the
predictive signal in this supply-chain task requires non-linear decision boundaries — a central
question in the comparison analysis of Phase 6.